# MDC Anomaly Detection — Model vNext (research-grade)


**Self-contained:** upload **only this notebook**. Drive I/O helpers are defined as a readable code cell (no base64 blobs, no extra `.py` upload).
**Drive:** `My Drive / vNEXT_test / {processed|runs}`
Loads `windows_vnext.npz` from `mdc_preprocess_vNext.ipynb`. Builds on the winning Exp A architecture (test ROC-AUC **0.7163**, MCC **0.5278** in `mdc_model_v3_output_4`) and addresses the **peak-then-decay** failure observed there (val AUC peaked at epoch 6 then collapsed by epoch 30 as the AE learned to reconstruct attacks too).

**New vs Exp A:**

| Lever | Change | Failure addressed |
|---|---|---|
| Aggressive AUC early stop | stop after 3 consecutive misses past epoch 8 | Peak-then-decay |
| Contractive-noise regularizer | `λ‖z(x+ε)−z(x)‖²` | Attack reconstruction creep |
| Smaller bottleneck default | `BOTTLENECK_DIM=24` (HPO sweep 16/24/32) | Same |
| **Ensemble of AUC-best + MSE-best** state-dicts | rank-average scoring | Single-checkpoint brittleness |
| Multi-seed evaluation | 3 seeds, report mean ± std | Single-run variance |
| F1-on-val threshold | primary operating point | High FPR at Youden |
| **Drift monitor** (PSI/KS) | holdout vs benign-train baseline | Thesis “drift-aware” claim |
| **Per-attack-type eval** (§19) | recall by MDC label 0–11 | Multiclass breakdown |

**Kept from Exp A:** Sequence bottleneck (no mean-pool), max-over-time scoring, AUC-best checkpoint, score-flip safety, MSE loss + denoising, ~166 features.

**Prerequisite for §19:** run `mdc_preprocess_vNext_mc.ipynb` first (`windows_vnext_mc.npz`).

**KAGGLE version:** no Google Drive, no Colab. All artifacts live under `/kaggle/working/`; hand off to the next notebook via '+ Add Data' or a downloaded/re-uploaded zip -- see `mdc_preprocess_vNext_kaggle.ipynb`'s title cell for the full pattern.


## 0. Environment

In [ ]:
import sys, os, subprocess
_IN_KAGGLE = os.path.exists('/kaggle/working')
if _IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'scikit-learn', 'matplotlib', 'optuna'], check=False)
print(f'In Kaggle: {_IN_KAGGLE}')


## 1. Configuration

In [ ]:
from __future__ import annotations
import os, json, gc, random, math
from pathlib import Path
import numpy as np

VERSION = 'vnext'
SEEDS   = [42, 7, 1337]
MAIN_SEED = SEEDS[0]
random.seed(MAIN_SEED); np.random.seed(MAIN_SEED)

BATCH_SIZE   = 32
MAX_EPOCHS   = 80
LR           = 5e-4
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.15
GRAD_CLIP    = 1.0

# Architecture (default; HPO can override)
D_MODEL        = 64
N_HEAD         = 4
N_ENC_LAYERS   = 2
N_DEC_LAYERS   = 2
DIM_FF         = 256
BOTTLENECK_DIM = 24            # smaller default than Exp A (32) to limit attack reconstruction

# Regularization (vNext)
USE_DENOISING       = True
NOISE_STD           = 0.03
CONTRACTIVE_LAMBDA  = 1e-3     # weight for ||z(x+e) - z(x)||^2
CONTRACTIVE_NOISE   = 0.03

# Scoring
SCORE_MEAN_W                 = 0.3
SCORE_MAX_W                  = 0.7
SCORE_BATCH_SIZE             = 128
AUC_CHECK_FREQ               = 2          # every 2 epochs (finer than Exp A)
AUC_RESTORE_MIN              = 0.50
USE_FEAT_VAR_WEIGHT_MIN_AUC  = 0.55
AUTO_SCORE_FLIP              = True

# Early stopping (key vNext change)
AUC_EARLY_STOP   = True
AUC_PATIENCE     = 3            # stop after 3 consecutive AUC misses
MIN_EPOCH_FOR_STOP = 8          # but only after epoch 8

# HPO
N_TRIALS      = 12
HPO_MAX_EPOCH = 18
HPO_PATIENCE  = 4

NPZ_FILE   = f'windows_{VERSION}.npz'
DRIFT_FILE = f'drift_baseline_{VERSION}.npz'

DATA_DIR = Path('/kaggle/working/processed') if _IN_KAGGLE else Path('../outputs/vNEXT_test/processed')
RUN_DIR  = Path('/kaggle/working/runs')      if _IN_KAGGLE else Path('../outputs/vNEXT_test/runs')
for _p in (DATA_DIR, RUN_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print('Config loaded')
print(f'  DATA_DIR (use)  : {DATA_DIR}')
print(f'  RUN_DIR  (use)  : {RUN_DIR}')
print(f'  ARCH            : D={D_MODEL} bn={BOTTLENECK_DIM} ff={DIM_FF} enc={N_ENC_LAYERS} dec={N_DEC_LAYERS}')
print(f'  early-stop      : AUC patience={AUC_PATIENCE} after epoch {MIN_EPOCH_FOR_STOP}')
print(f'  contractive     : lambda={CONTRACTIVE_LAMBDA} noise={CONTRACTIVE_NOISE}')
print(f'  scoring         : mean_w={SCORE_MEAN_W}  max_w={SCORE_MAX_W}  auto_flip={AUTO_SCORE_FLIP}')

## 2. Mount drive + load npz + scale gate

In [ ]:
# --- Kaggle IO helpers (self-contained -- no Drive, no Colab) ---
import os, sys, glob, shutil, zipfile, base64
from pathlib import Path

_IN_KAGGLE = os.path.exists('/kaggle/working')

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')

def kaggle_find(name, extra_dirs=()):
    """Search /kaggle/input/**, /kaggle/working/**, and extra_dirs for a file by
    name. There is no live shared Drive on Kaggle -- to hand a file from one
    notebook to the next, either (a) attach the producing notebook's own
    output via '+ Add Data > Your Notebooks' (no manual zip needed), or
    (b) download this notebook's output zip and upload it as a new Kaggle
    Dataset, then attach that dataset. Either way it shows up under
    /kaggle/input/<name>/ and this function finds it there."""
    roots = [KAGGLE_INPUT, KAGGLE_WORKING, *[Path(d) for d in extra_dirs]]
    for root in roots:
        if not root.is_dir():
            continue
        hits = sorted(glob.glob(str(root / '**' / name), recursive=True))
        if hits:
            return Path(hits[0])
    return None

def kaggle_upload_fallback(name, dest_dir):
    """Best-effort interactive upload widget for a live session. Not available
    during a headless 'Save & Run All' commit (no UI) -- attach the file as
    input data instead in that case."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
        uploader = widgets.FileUpload(accept='', multiple=False)
        display(uploader)
        print(f'Upload {name} with the widget above, then re-run this cell.')
        if uploader.value:
            item = list(uploader.value.values())[0]
            content = item['content'] if isinstance(item, dict) else item.content
            dest = Path(dest_dir) / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(bytes(content))
            print(f'Saved -> {dest}')
            return dest
    except Exception as e:
        print(f'Interactive upload unavailable ({e}).')
    return None

def zip_and_offer_download(src_dir, zip_name, max_auto_mb=25):
    """Zip src_dir into /kaggle/working/{zip_name}.zip and try to trigger a
    browser download automatically. Only fires in a live, actively-open
    browser tab (not during headless 'Save & Run All') -- Kaggle's own Output
    tab (right sidebar) always lists this zip for manual download regardless
    of whether the auto-download trick actually fires in your browser."""
    src_dir = Path(src_dir)
    zip_base = KAGGLE_WORKING / zip_name
    zip_path = zip_base.with_suffix('.zip')
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_base), 'zip', root_dir=src_dir)
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f'Zipped -> {zip_path}  ({size_mb:.1f} MB)')
    if size_mb <= max_auto_mb:
        try:
            from IPython.display import HTML, display
            b64 = base64.b64encode(zip_path.read_bytes()).decode()
            html = (
                f'<a id="dl_{zip_name}" download="{zip_path.name}" '
                f'href="data:application/zip;base64,{b64}"></a>'
                f'<script>document.getElementById("dl_{zip_name}").click();</script>'
            )
            display(HTML(html))
            print('Auto-download triggered (only works if this tab is actively open --')
            print('if nothing happened, use the Output tab on the right instead).')
        except Exception as e:
            print(f'Auto-download trick failed ({e}) -- use the Output tab instead.')
    else:
        print(f'{size_mb:.1f} MB exceeds the {max_auto_mb} MB auto-download guard -- '
              'skipping the browser trick to avoid bloating notebook output.')
        print('Get it from the Output tab (right sidebar) after Save Version instead.')
    return zip_path

print(f'Kaggle IO ready. In Kaggle: {_IN_KAGGLE}')

NPZ_FILE   = f'windows_{VERSION}.npz'
DRIFT_FILE = f'drift_baseline_{VERSION}.npz'
NPZ_CANDIDATES = [NPZ_FILE, 'windows_vnext.npz', 'windows_vnext_mc.npz']

hit = None
for cand in NPZ_CANDIDATES:
    hit = kaggle_find(cand)
    if hit is not None:
        NPZ_FILE = hit.name
        break

if hit is None:
    print(f'None of {NPZ_CANDIDATES} found under /kaggle/input or /kaggle/working.')
    print("Attach mdc_preprocess_vNext_kaggle.ipynb's output via '+ Add Data', or upload it now:")
    hit = kaggle_upload_fallback(NPZ_FILE, DATA_DIR)

if hit is None:
    raise FileNotFoundError(
        f'Cannot find any of {NPZ_CANDIDATES} under /kaggle/input or /kaggle/working. '
        "Run mdc_preprocess_vNext_kaggle.ipynb first and attach its output via "
        "'+ Add Data', or upload the npz manually."
    )

NPZ_PATH = DATA_DIR / hit.name
if hit.resolve() != NPZ_PATH.resolve():
    shutil.copy2(hit, NPZ_PATH)

drift_hit = kaggle_find(DRIFT_FILE)
DRIFT_PATH = DATA_DIR / DRIFT_FILE
if drift_hit is not None and drift_hit.resolve() != DRIFT_PATH.resolve():
    shutil.copy2(drift_hit, DRIFT_PATH)
elif drift_hit is None:
    print(f'WARN: drift baseline not found yet at {DRIFT_PATH}')

print(f'NPZ_PATH   : {NPZ_PATH} ({NPZ_PATH.stat().st_size:,} bytes)')
print(f'DRIFT_PATH : {DRIFT_PATH} exists={DRIFT_PATH.is_file()}')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'RUN_DIR    : {RUN_DIR}')

# ---------------------------------------------------------------------------
# Load windows npz + scale gate (defines X_train / X_val / X_test / y_*)
# ---------------------------------------------------------------------------
z = np.load(NPZ_PATH, allow_pickle=True)
X_train = z['X_train'].astype(np.float32)
X_val   = z['X_val'].astype(np.float32)
X_test  = z['X_test'].astype(np.float32)
y_val   = z['y_val'].astype(np.int64)
y_test  = z['y_test'].astype(np.int64)
c_val   = z['c_val']  if 'c_val'  in z.files else None
c_test  = z['c_test'] if 'c_test' in z.files else None
ts_val  = z['ts_val'].astype(np.int64)  if 'ts_val'  in z.files else None
ts_test = z['ts_test'].astype(np.int64) if 'ts_test' in z.files else None
z.close()

n_features = X_train.shape[2]
T          = X_train.shape[1]
_xmax  = float(max(np.abs(X_train).max(), np.abs(X_val).max(), np.abs(X_test).max()))
_xmean = float(np.abs(X_train).mean())
_nan   = bool(np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any())
DATA_SCALE_OK = (_xmax <= 50 and _xmean <= 20 and not _nan)
if not DATA_SCALE_OK:
    raise ValueError(f'BAD npz scale: max|X|={_xmax:.3e} mean|X|={_xmean:.3e} nan={_nan}')

print(f'X_train : {X_train.shape}  (benign only)')
print(f'X_val   : {X_val.shape}    attack {y_val.mean()*100:.1f}%')
print(f'X_test  : {X_test.shape}   attack {y_test.mean()*100:.1f}%')
print(f'T={T}  n_features={n_features}  mean|X|={_xmean:.3f}  max|X|={_xmax:.3f}')
if ts_test is not None:
    print(f'ts_test : [{int(ts_test.min())}, {int(ts_test.max())}]')

_drift = Path(DRIFT_PATH) if DRIFT_PATH is not None else (LOCAL_DATA_CACHE / DRIFT_FILE)
if _drift.is_file():
    drift_z = np.load(_drift, allow_pickle=True)
    print(f'Drift baseline loaded ({drift_z["mean"].shape[0]} features) <- {_drift}')
else:
    drift_z = None
    print(f'Drift baseline NOT found at {_drift} — drift monitor will be skipped')


## 3. Tensors / loaders / device

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

X_train_t = torch.from_numpy(X_train).float()
X_val_t   = torch.from_numpy(X_val).float()
train_loader_base = DataLoader(
    TensorDataset(X_train_t),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
    pin_memory=False, num_workers=0,
)
print(f'Train batches: {len(train_loader_base)}  val windows: {len(X_val_t)}  test windows: {len(X_test)}')

## 4. Scoring functions (defined before model so training can call them)

Score = `mean_w * mean(err) + max_w * max_over_t(mean_over_F(err))`. `feat_std` divides per-feature error after training and is only applied when val AUC ≥ `USE_FEAT_VAR_WEIGHT_MIN_AUC`.

In [ ]:
from sklearn.metrics import roc_auc_score as _roc_auc_score

SCORE_INVERT = False

def _combine(err, mean_w=SCORE_MEAN_W, max_w=SCORE_MAX_W):
    s_mean = err.mean(dim=(1, 2))
    s_max  = err.mean(dim=2).amax(dim=1)
    return mean_w * s_mean + max_w * s_max

@torch.no_grad()
def _eval_auc(m, X_t, y, dev=device, bs=SCORE_BATCH_SIZE):
    m.eval()
    out = []
    for k in range(0, len(X_t), bs):
        xb = X_t[k:k+bs].to(dev)
        rb = m(xb)
        err = (rb - xb).pow(2)
        out.append(_combine(err).cpu().numpy())
        del xb, rb, err
    s = np.concatenate(out)
    try:
        r = float(_roc_auc_score(y, s))
        f = float(_roc_auc_score(y, -s))
        return max(r, f)
    except Exception:
        return 0.5

@torch.no_grad()
def compute_scores(X, feat_std=None, _model=None, mean_w=SCORE_MEAN_W, max_w=SCORE_MAX_W,
                   bs=SCORE_BATCH_SIZE, invert=None):
    m = _model if _model is not None else model
    m.eval()
    if invert is None: invert = SCORE_INVERT
    out = []
    for k in range(0, len(X), bs):
        xb = torch.from_numpy(X[k:k+bs]).float().to(device)
        rb = m(xb)
        err = (rb - xb).pow(2)
        if feat_std is not None:
            w = torch.from_numpy(feat_std).float().to(device)
            err = err / w[None, None, :]
            del w
        out.append(_combine(err, mean_w, max_w).cpu().numpy())
        del xb, rb, err
    s = np.concatenate(out)
    return -s if invert else s

def inversion_msg():
    return ('Scale OK -- inversion is model/representation, not preprocess. '
            'Verify Tier 1 arch (no mean-pool) and ~166 features.')

def score_with_protocol(m, X_val, y_val, X_test, feat_var_min_auc=USE_FEAT_VAR_WEIGHT_MIN_AUC):
    """Unified scoring protocol (ISS-04 fix).

    Same invert-flip decision + feat_std gating for the default, multi-seed,
    and HPO paths, so cross-experiment ROC-AUC comparisons are apples-to-apples.
    Previously only the default path (cell-score-flip) computed feat_std;
    multi-seed (cell-seeds) and HPO (cell-hpo-retrain) always passed
    feat_std=None, silently making the default numbers not directly
    comparable to the other two. Ensemble (Sec 9) stays default-path-only
    by design -- running it per seed / per HPO trial would multiply
    training cost without changing what those experiments measure.
    """
    # invert=False is explicit here (not left to default None) so this
    # direction-decision step never inherits the module-level SCORE_INVERT
    # global left behind by whichever model was scored before this one --
    # that cross-contamination is what silently flipped multi-seed/HPO
    # scores to the wrong sign when they ran after the default path.
    s_val_raw  = compute_scores(X_val,  feat_std=None, _model=m, invert=False)
    s_test_raw = compute_scores(X_test, feat_std=None, _model=m, invert=False)
    val_raw  = float(_roc_auc_score(y_val,  s_val_raw))
    val_flip = float(_roc_auc_score(y_val, -s_val_raw))
    invert = bool(AUTO_SCORE_FLIP and val_flip > val_raw)

    feat_std_m = None
    if max(val_raw, val_flip) >= feat_var_min_auc:
        Xb = X_val[y_val == 0]
        m.eval(); errs = []
        with torch.no_grad():
            for k in range(0, len(Xb), SCORE_BATCH_SIZE):
                xb = torch.from_numpy(Xb[k:k+SCORE_BATCH_SIZE]).float().to(device)
                errs.append(((m(xb) - xb).pow(2)).cpu().numpy())
                del xb
        e = np.concatenate(errs, axis=0).reshape(-1, n_features)
        feat_std_m = e.std(axis=0).astype(np.float32) + 1e-6
        del errs, e, Xb; gc.collect()

    s_val  = compute_scores(X_val,  feat_std=feat_std_m, _model=m, invert=invert)
    s_test = compute_scores(X_test, feat_std=feat_std_m, _model=m, invert=invert)
    return {
        's_val': s_val, 's_test': s_test, 'invert': invert,
        'feat_std': feat_std_m, 'feat_std_used': feat_std_m is not None,
        'val_auc_raw': val_raw, 'val_auc_flip': val_flip,
        's_val_raw': s_val_raw, 's_test_raw': s_test_raw,
    }

print('scoring functions ready (score_with_protocol: ISS-04 unified default/multi-seed/HPO scoring)')

## 5. Model — Sequence bottleneck Transformer AE (no temporal mean-pool)
Identical encoder/decoder layout to the Exp A winning model. The encoder produces a per-timestep bottleneck `(B, T, bn)`; the decoder uses learned position queries with that bottleneck as memory — so reconstruction must pass through a low-rank temporal channel without leaking the input.

In [ ]:
import torch.nn as nn

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SequenceBottleneckAE(nn.Module):
    def __init__(self, n_features, d_model=D_MODEL, nhead=N_HEAD,
                 num_enc_layers=N_ENC_LAYERS, num_dec_layers=N_DEC_LAYERS,
                 dim_ff=DIM_FF, dropout=DROPOUT, bottleneck_dim=BOTTLENECK_DIM,
                 max_len=200):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pe         = SinusoidalPE(d_model, max_len=max_len, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)
        self.bn_down = nn.Sequential(
            nn.Linear(d_model, bottleneck_dim),
            nn.LayerNorm(bottleneck_dim), nn.GELU())
        self.bn_up   = nn.Linear(bottleneck_dim, d_model)
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_dec_layers)
        self.output_proj = nn.Linear(d_model, n_features)
        self.pos_queries = nn.Parameter(torch.zeros(max_len, d_model))
        self.d_model = d_model
        self.bottleneck_dim = bottleneck_dim
    def encode(self, x):
        return self.bn_down(self.encoder(self.pe(self.input_proj(x))))
    def forward(self, x):
        B, T, _ = x.shape
        z      = self.encode(x)
        memory = self.bn_up(z)
        tgt    = self.pe(self.pos_queries[:T].unsqueeze(0).expand(B, -1, -1))
        return self.output_proj(self.decoder(tgt, memory))

def build_model(n_features, d_model=D_MODEL, nhead=N_HEAD,
                num_enc_layers=N_ENC_LAYERS, num_dec_layers=N_DEC_LAYERS,
                dim_ff=DIM_FF, dropout=DROPOUT, bottleneck_dim=BOTTLENECK_DIM):
    return SequenceBottleneckAE(
        n_features=n_features, d_model=d_model, nhead=nhead,
        num_enc_layers=num_enc_layers, num_dec_layers=num_dec_layers,
        dim_ff=dim_ff, dropout=dropout, bottleneck_dim=bottleneck_dim,
        max_len=max(T + 10, 200),
    ).to(device)

model = build_model(n_features)
print(f'Model: SequenceBottleneckAE  params: {sum(p.numel() for p in model.parameters()):,}')

## 6. Pre-train diagnostics (Tier 4)
PCA-PC1 separability AUC tells us whether benign and attack classes are linearly separable in raw window space. Untrained-AUC should be ~0.5 — large deviations are a scoring or model bug.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
n_pca = min(2000, len(X_val))
idx = np.random.default_rng(MAIN_SEED).choice(len(X_val), n_pca, replace=False)
Xflat = X_val[idx].reshape(n_pca, -1)
pca = PCA(n_components=2, random_state=MAIN_SEED).fit_transform(Xflat)
if len(np.unique(y_val[idx])) == 2:
    a = roc_auc_score(y_val[idx], pca[:, 0]); a = max(a, 1 - a)
    print(f'PCA-PC1 separability AUC : {a:.4f}')
_m0 = build_model(n_features)
n_d = min(512, len(X_val))
_s0 = compute_scores(X_val[:n_d], _model=_m0)
try:
    a0 = roc_auc_score(y_val[:n_d], _s0)
    print(f'Untrained model AUC      : {a0:.4f}  (raw; sign-agnostic max={max(a0,1-a0):.4f})')
except Exception as e:
    print('Untrained AUC failed:', e)
del _m0, _s0, Xflat, pca, idx; gc.collect()

## 7. Training: MSE + denoising + contractive-noise reg + AUC early stop
The contractive term `λ‖z(x+ε)−z(x)‖²` (Rifai et al. 2011 contractive-AE idea, computed via stochastic finite difference) pushes the encoder to be locally stable on benign inputs. The denoising term forces reconstruction from a noisy benign manifold. Together they slow the “attack reconstruction creep” that caused AUC decay after epoch 6 in Exp A.

In [ ]:
import torch.nn.functional as F
from torch import optim

def train_one(model, loader, X_val_t, y_val,
              max_epochs=MAX_EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
              grad_clip=GRAD_CLIP, use_denoising=USE_DENOISING, noise_std=NOISE_STD,
              contractive_lambda=CONTRACTIVE_LAMBDA, contractive_noise=CONTRACTIVE_NOISE,
              auc_check_freq=AUC_CHECK_FREQ, auc_patience=AUC_PATIENCE,
              min_epoch=MIN_EPOCH_FOR_STOP, auc_early_stop=None, verbose=True):
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=6, min_lr=1e-6)
    _early_stop = AUC_EARLY_STOP if auc_early_stop is None else bool(auc_early_stop)
    xv_benign = X_val_t[y_val == 0].to(device)
    if not len(xv_benign): xv_benign = X_val_t.to(device)
    @torch.no_grad()
    def benign_mse():
        model.eval(); return float(F.mse_loss(model(xv_benign), xv_benign).item())
    best_auc_state, best_mse_state = None, None
    best_auc, best_mse = 0.0, float('inf')
    no_improve_auc = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train(); running, nb = 0.0, 0
        for (xb,) in loader:
            xb = xb.to(device)
            xb_in = xb + torch.randn_like(xb) * noise_std if use_denoising else xb
            opt.zero_grad(set_to_none=True)
            recon = model(xb_in)
            loss = F.mse_loss(recon, xb)
            if contractive_lambda > 0:
                z1 = model.encode(xb_in)
                z2 = model.encode(xb_in + torch.randn_like(xb_in) * contractive_noise)
                loss = loss + contractive_lambda * F.mse_loss(z2, z1)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            running += float(loss.item()); nb += 1
        train_loss = running / max(nb, 1)
        vmse = benign_mse(); sch.step(vmse)
        cur_lr = opt.param_groups[0]['lr']
        row = {'epoch': epoch, 'train_loss': train_loss, 'val_mse_benign': vmse, 'lr': cur_lr}
        if vmse < best_mse - 1e-6:
            best_mse = vmse
            best_mse_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        auc = None
        if epoch % auc_check_freq == 0:
            auc = _eval_auc(model, X_val_t, y_val)
            row['val_auc'] = auc
            if auc > best_auc + 0.002:
                best_auc = auc; no_improve_auc = 0
                best_auc_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            elif epoch >= min_epoch:
                no_improve_auc += 1
        history.append(row)
        if verbose and (epoch == 1 or epoch % 2 == 0 or auc is not None):
            tag = f'  val_auc={auc:.4f}' if auc is not None else ''
            print(f'epoch {epoch:03d}  train={train_loss:.5f}  val_benign={vmse:.5f}  lr={cur_lr:.1e}{tag}')
        if _early_stop and epoch >= min_epoch and no_improve_auc >= auc_patience:
            print(f'early stop @ epoch {epoch}  best_auc={best_auc:.4f}  best_mse={best_mse:.5f}')
            break
    restored = 'last'
    if best_auc_state is not None:
        model.load_state_dict(best_auc_state); restored = f'AUC ({best_auc:.4f})'
    elif best_mse_state is not None:
        model.load_state_dict(best_mse_state); restored = f'MSE ({best_mse:.5f})'
    print(f'restored: {restored}')
    return ({'best_val_mse': best_mse, 'best_val_auc': best_auc,
             'final_epoch': epoch, 'restored': restored,
             'auc_state': best_auc_state, 'mse_state': best_mse_state},
            history)

train_result, history = train_one(model, train_loader_base, X_val_t, y_val)

## 8. Scoring + auto score-flip + per-feature variance weighting

In [ ]:
# Unified scoring protocol (ISS-04 fix): the default path now shares the
# exact same invert-flip + feat_std logic as multi-seed (cell-seeds) and
# HPO (cell-hpo-retrain) via score_with_protocol(), instead of each path
# duplicating its own (previously divergent) version of this logic.
_proto = score_with_protocol(model, X_val, y_val, X_test)
SCORE_INVERT = _proto['invert']
feat_std = _proto['feat_std']
s_val, s_test = _proto['s_val'], _proto['s_test']

test_raw  = float(roc_auc_score(y_test,  _proto['s_test_raw']))
test_flip = float(roc_auc_score(y_test, -_proto['s_test_raw']))
print(f"val  AUC raw={_proto['val_auc_raw']:.4f} flip={_proto['val_auc_flip']:.4f}  -> invert={SCORE_INVERT}")
print(f'test AUC raw={test_raw:.4f} flip={test_flip:.4f}')

if _proto['feat_std_used']:
    print(f"feat_std applied  (min={feat_std.min():.4f}  max={feat_std.max():.4f})")
else:
    print(f"feat_std skipped (max val AUC {max(_proto['val_auc_raw'], _proto['val_auc_flip']):.4f} < {USE_FEAT_VAR_WEIGHT_MIN_AUC})")

## 9. Ensemble of AUC-best + MSE-best checkpoints
Average ranks of scores from the two saved state-dicts. Reduces single-checkpoint variance — in Exp A the AUC-best (epoch 6) was strong but unique; if it had been one bad epoch away, the run would have collapsed.

In [ ]:
from scipy.stats import rankdata

def scores_with_state(state, X):
    m = build_model(n_features); m.load_state_dict(state); m.to(device).eval()
    s = compute_scores(X, feat_std=feat_std, _model=m)
    del m
    return s

use_ens = (train_result['auc_state'] is not None) and (train_result['mse_state'] is not None)
if use_ens:
    s_val_a  = scores_with_state(train_result['auc_state'], X_val)
    s_val_m  = scores_with_state(train_result['mse_state'], X_val)
    s_test_a = scores_with_state(train_result['auc_state'], X_test)
    s_test_m = scores_with_state(train_result['mse_state'], X_test)
    s_val_ens  = 0.5 * (rankdata(s_val_a)  + rankdata(s_val_m))
    s_test_ens = 0.5 * (rankdata(s_test_a) + rankdata(s_test_m))
    auc_v_ens = roc_auc_score(y_val,  s_val_ens)
    auc_t_ens = roc_auc_score(y_test, s_test_ens)
    auc_v_a   = roc_auc_score(y_val, s_val_a)
    auc_t_a   = roc_auc_score(y_test, s_test_a)
    print(f'AUC-best ckpt only -- val={auc_v_a:.4f}  test={auc_t_a:.4f}')
    print(f'Ensemble (rank avg)-- val={auc_v_ens:.4f}  test={auc_t_ens:.4f}')
    if auc_v_ens >= auc_v_a:
        s_val, s_test = s_val_ens, s_test_ens
        print('Using ENSEMBLE for downstream thresholds/metrics')
    else:
        s_val, s_test = s_val_a, s_test_a
        print('Using AUC-best (ensemble did not improve val AUC)')
else:
    print('Ensemble unavailable -- only one checkpoint exists; keep current scores')

## 10. Threshold selection
Default operating point switched to **F1-on-val** (lower FPR than Youden, comparable recall).

In [ ]:
from sklearn.metrics import (roc_curve, precision_recall_curve,
    confusion_matrix, average_precision_score, f1_score,
    matthews_corrcoef, balanced_accuracy_score)

def thresholds_from_val(s_val, y_val):
    out = {}
    fpr_r, tpr_r, thr_r = roc_curve(y_val, s_val)
    out['youden_j'] = float(thr_r[int(np.argmax(tpr_r[1:] - fpr_r[1:]))])
    s_b = s_val[y_val == 0]
    out['p95_benign'] = float(np.percentile(s_b, 95))
    out['p99_benign'] = float(np.percentile(s_b, 99))
    prec, rec, thr_pr = precision_recall_curve(y_val, s_val)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    out['f1_optimal'] = float(thr_pr[int(np.argmax(f1))])
    fb = (1.25) * prec[:-1] * rec[:-1] / (0.25 * prec[:-1] + rec[:-1] + 1e-9)
    out['fbeta_0.5'] = float(thr_pr[int(np.argmax(fb))])
    return out

thresholds = thresholds_from_val(s_val, y_val)
print('thresholds:', json.dumps({k: round(v, 4) for k, v in thresholds.items()}))

## 11. Full test evaluation

In [ ]:
def full_eval(s, y, threshold):
    pred = (s >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return dict(
        threshold=float(threshold),
        roc_auc=float(roc_auc_score(y, s)), pr_auc=float(average_precision_score(y, s)),
        f1=float(f1_score(y, pred, zero_division=0)),
        mcc=float(matthews_corrcoef(y, pred)),
        bal_acc=float(balanced_accuracy_score(y, pred)),
        precision=float(tp/(tp+fp)) if tp+fp>0 else 0.0,
        recall   =float(tp/(tp+fn)) if tp+fn>0 else 0.0,
        fpr=float(fp/(fp+tn)) if fp+tn>0 else 0.0,
        fnr=float(fn/(fn+tp)) if fn+tp>0 else 0.0,
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))

print(f'{"strategy":<14} {"ROC":>7} {"PR":>7} {"F1":>7} {"MCC":>7} {"BalAcc":>7} {"FPR":>7} {"FNR":>7}')
print('-'*72)
all_results = {}
for name, thr in thresholds.items():
    r = full_eval(s_test, y_test, thr); all_results[name] = r
    print(f'{name:<14} {r["roc_auc"]:>7.4f} {r["pr_auc"]:>7.4f} {r["f1"]:>7.4f}'
          f' {r["mcc"]:>7.4f} {r["bal_acc"]:>7.4f} {r["fpr"]:>7.4f} {r["fnr"]:>7.4f}')

primary = all_results['f1_optimal']
print(f'\nPRIMARY (f1_optimal): ROC={primary["roc_auc"]:.4f}  PR={primary["pr_auc"]:.4f}  '
      f'F1={primary["f1"]:.4f}  MCC={primary["mcc"]:.4f}')

## 12. Score-distribution + ROC/PR plots

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
s_bn = s_test[y_test == 0]; s_at = s_test[y_test == 1]
hi = np.percentile(s_test, 99); bins = np.linspace(np.percentile(s_test, 1), hi, 80)
ax[0].hist(s_bn, bins=bins, alpha=0.6, label=f'benign n={len(s_bn)}', density=True)
ax[0].hist(s_at, bins=bins, alpha=0.6, label=f'attack n={len(s_at)}', density=True)
for k in ('youden_j','f1_optimal'):
    if k in thresholds: ax[0].axvline(thresholds[k], ls='--', lw=1, label=k)
ax[0].set_title('Test score distribution'); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for split, s, y, ls in [('val', s_val, y_val, '-'), ('test', s_test, y_test, '--')]:
    f, t, _ = roc_curve(y, s)
    ax[1].plot(f, t, ls=ls, label=f'{split} AUC={roc_auc_score(y, s):.4f}')
ax[1].plot([0,1],[0,1],'k:',lw=0.8); ax[1].set_title('ROC'); ax[1].grid(alpha=0.3); ax[1].legend()
for split, s, y, ls in [('val', s_val, y_val, '-'), ('test', s_test, y_test, '--')]:
    p, r, _ = precision_recall_curve(y, s)
    ax[2].plot(r, p, ls=ls, label=f'{split} AP={average_precision_score(y, s):.4f}')
ax[2].set_title('Precision-Recall'); ax[2].grid(alpha=0.3); ax[2].legend()
plt.tight_layout(); plt.savefig(RUN_DIR / 'eval_vnext.png', dpi=140); plt.show()

## 13. Multi-seed evaluation (variance control)
Retrain with seeds [42, 7, 1337] using the default config and report mean ± std for the F1-optimal threshold. Demonstrates that the headline number is not a single lucky seed.

In [ ]:
def run_single_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = build_model(n_features)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=False, pin_memory=False, num_workers=0)
    tr, _ = train_one(m, loader, X_val_t, y_val, verbose=False)
    # Unified scoring protocol (ISS-04): same invert-flip + feat_std policy
    # as the default path (cell-score-flip) via score_with_protocol().
    _proto = score_with_protocol(m, X_val, y_val, X_test)
    sv, st = _proto['s_val'], _proto['s_test']
    thr = thresholds_from_val(sv, y_val)
    r   = full_eval(st, y_test, thr['f1_optimal'])
    del m, loader; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return r, tr['best_val_auc'], tr['final_epoch']

seed_results = []
for s in SEEDS:
    r, bv, ep = run_single_seed(s)
    print(f'seed={s:>5}  best_val_auc={bv:.4f}  epoch={ep:>3}  '
          f'test ROC={r["roc_auc"]:.4f}  F1={r["f1"]:.4f}  MCC={r["mcc"]:.4f}')
    seed_results.append({'seed': s, 'best_val_auc': bv, 'final_epoch': ep, **r})

agg_keys = ['roc_auc', 'pr_auc', 'f1', 'mcc', 'bal_acc', 'fpr', 'fnr']
agg = {k: (float(np.mean([r[k] for r in seed_results])),
           float(np.std ([r[k] for r in seed_results]))) for k in agg_keys}
print('\nmulti-seed summary (mean +/- std)')
for k, (m_, s_) in agg.items(): print(f'  {k:<8} {m_:.4f} +/- {s_:.4f}')

## 14. Drift monitor (PSI + KS on holdout vs benign train baseline)
Reads `drift_baseline_vnext.npz`. Reports per-feature PSI and KS distances on the (test) windows vs the training benign baseline. Establishes the foundation for an automated retrain trigger (PSI > 0.25 = major drift).

In [ ]:
if drift_z is None:
    print('No drift baseline -- skip')
else:
    q_levels = drift_z['q_levels']
    q_base   = drift_z['q']            # (Q, F)
    def psi_per_feature(X, bins_per_feat):
        F_ = X.shape[2]
        flat = X.reshape(-1, F_)
        psis = np.zeros(F_, dtype=np.float32)
        for j in range(F_):
            edges = np.unique(bins_per_feat[:, j])
            if len(edges) < 3:
                psis[j] = 0.0; continue
            edges[0] = -np.inf; edges[-1] = np.inf
            base_hist, _ = np.histogram(bins_per_feat[:, j], bins=edges)
            curr_hist, _ = np.histogram(flat[:, j],          bins=edges)
            base_p = (base_hist + 1) / (base_hist.sum() + len(edges) - 1)
            curr_p = (curr_hist + 1) / (curr_hist.sum() + len(edges) - 1)
            psis[j] = float(np.sum((curr_p - base_p) * np.log(curr_p / base_p)))
        return psis
    def ks_per_feature(X, base_q, q_levels):
        F_ = X.shape[2]
        flat = X.reshape(-1, F_)
        ks = np.zeros(F_, dtype=np.float32)
        for j in range(F_):
            curr_q = np.quantile(flat[:, j], q_levels)
            base   = base_q[:, j]
            ks[j]  = float(np.max(np.abs(curr_q - base)))
        return ks
    # use baseline q array as the histogram edges (per feature)
    psi = psi_per_feature(X_test, q_base)
    ks  = ks_per_feature (X_test, q_base, q_levels)
    n_major = int((psi > 0.25).sum()); n_minor = int(((psi > 0.10) & (psi <= 0.25)).sum())
    print(f'PSI features: major (>0.25)={n_major}   moderate (0.10-0.25)={n_minor}')
    print(f'KS  median={float(np.median(ks)):.4f}  max={float(ks.max()):.4f}')
    top_psi = np.argsort(psi)[::-1][:8]
    print('top PSI features:')
    for j in top_psi: print(f'  idx={int(j):>4}  PSI={psi[j]:.3f}  KS={ks[j]:.3f}')
    np.savez_compressed(RUN_DIR / 'drift_report_vnext.npz', psi=psi, ks=ks)

## 15. Optuna HPO (compact)
Bottleneck swept in {16, 24, 32}. Objective = max(val raw AUC, val flip AUC) to make HPO robust to direction. Best params are then retrained with the full pipeline.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def hpo_objective(trial):
    cfg = dict(
        d_model        = trial.suggest_categorical('d_model',        [64, 96]),
        bottleneck_dim = trial.suggest_categorical('bottleneck_dim', [16, 24, 32]),
        num_enc_layers = trial.suggest_int('num_enc_layers', 1, 3),
        num_dec_layers = trial.suggest_int('num_dec_layers', 1, 2),
        dim_ff         = trial.suggest_categorical('dim_ff', [192, 256, 320]),
        dropout        = trial.suggest_float('dropout', 0.05, 0.30),
    )
    lr        = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    noise_std = trial.suggest_float('noise_std', 0.01, 0.06)
    cl        = trial.suggest_float('contractive_lambda', 0.0, 5e-3)
    nhead = 4 if cfg['d_model'] % 4 == 0 else 2
    m = build_model(n_features, d_model=cfg['d_model'], nhead=nhead,
                    num_enc_layers=cfg['num_enc_layers'],
                    num_dec_layers=cfg['num_dec_layers'],
                    dim_ff=cfg['dim_ff'], dropout=cfg['dropout'],
                    bottleneck_dim=cfg['bottleneck_dim'])
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE,
                        shuffle=True, drop_last=False, pin_memory=False, num_workers=0)
    res, _ = train_one(m, loader, X_val_t, y_val,
                       max_epochs=HPO_MAX_EPOCH, lr=lr,
                       noise_std=noise_std, contractive_lambda=cl,
                       auc_patience=HPO_PATIENCE, min_epoch=4, verbose=False)
    val_auc = res['best_val_auc']
    del m, loader; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return float(val_auc)

study = optuna.create_study(direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=MAIN_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
print(f'Starting HPO: {N_TRIALS} trials, {HPO_MAX_EPOCH} epochs each')
study.optimize(hpo_objective, n_trials=N_TRIALS, show_progress_bar=True)
best = study.best_trial
print(f'Best Val AUC: {best.value:.4f}')
print(f'Best params : {best.params}')

## 16. Retrain with best HPO params + full evaluation

In [ ]:
bp = best.params
nhead_best = 4 if bp['d_model'] % 4 == 0 else 2
model_best = build_model(
    n_features=n_features, d_model=bp['d_model'], nhead=nhead_best,
    num_enc_layers=bp['num_enc_layers'], num_dec_layers=bp['num_dec_layers'],
    dim_ff=bp['dim_ff'], dropout=bp['dropout'],
    bottleneck_dim=bp['bottleneck_dim'])
loader_best = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE,
                         shuffle=True, drop_last=False, pin_memory=False, num_workers=0)
tr_best, _ = train_one(model_best, loader_best, X_val_t, y_val,
                       lr=bp['lr'], noise_std=bp['noise_std'],
                       contractive_lambda=bp['contractive_lambda'],
                       verbose=True)

# Unified scoring protocol (ISS-04): same invert-flip + feat_std policy as
# the default path (cell-score-flip) via score_with_protocol().
_proto_hpo = score_with_protocol(model_best, X_val, y_val, X_test)
sv, st = _proto_hpo['s_val'], _proto_hpo['s_test']
thr_best = thresholds_from_val(sv, y_val)
r_best = full_eval(st, y_test, thr_best['f1_optimal'])
print('HPO best on test (F1-optimal threshold):')
for k in ('roc_auc','pr_auc','f1','mcc','bal_acc','fpr','fnr'):
    print(f'  {k:<8} {r_best[k]:.4f}')
print(f"HPO scoring protocol: invert={_proto_hpo['invert']}  feat_std_used={_proto_hpo['feat_std_used']}")

## 17. Save artifacts + metrics summary

In [ ]:
from datetime import datetime, timezone

torch.save({
    'model_default': model.state_dict(),
    'model_best'   : model_best.state_dict(),
    'hpo_params'   : bp,
    'feat_std'     : feat_std,
    'config': {
        'T': T, 'n_features': n_features,
        'd_model': D_MODEL, 'bottleneck_dim': BOTTLENECK_DIM,
        'nhead': N_HEAD, 'num_enc_layers': N_ENC_LAYERS,
        'num_dec_layers': N_DEC_LAYERS, 'dim_ff': DIM_FF, 'dropout': DROPOUT,
    },
}, RUN_DIR / 'checkpoint_vnext.pt')

np.savez_compressed(RUN_DIR / 'scores_vnext.npz',
    s_val=s_val, s_test=s_test, y_val=y_val, y_test=y_test)

summary = {
    'created_at'     : datetime.now(timezone.utc).isoformat(),
    'version'        : VERSION,
    'architecture'   : 'SequenceBottleneckAE (no temporal mean-pool)',
    'auto_score_flip': bool(SCORE_INVERT),
    'primary_threshold': 'f1_optimal',
    'test_default'   : all_results,
    'test_hpo'       : r_best,
    'multi_seed'     : {k: {'mean': v[0], 'std': v[1]} for k, v in agg.items()},
    'hpo_best_params': bp,
    'data_scale'     : {'mean_abs': _xmean, 'max_abs': _xmax, 'nan': _nan},
    'train_shapes'   : {'X_train': list(X_train_t.shape),
                        'X_val'  : list(X_val.shape),
                        'X_test' : list(X_test.shape)},
}
(RUN_DIR / 'metrics_vnext.json').write_text(json.dumps(summary, indent=2))
print('Saved artifacts to', RUN_DIR)
print(json.dumps({k: round(v, 4) for k, v in {
    'default_test_roc' : all_results['f1_optimal']['roc_auc'],
    'default_test_f1'  : all_results['f1_optimal']['f1'],
    'default_test_mcc' : all_results['f1_optimal']['mcc'],
    'hpo_test_roc'     : r_best['roc_auc'],
    'hpo_test_f1'      : r_best['f1'],
    'hpo_test_mcc'     : r_best['mcc'],
    'mean_seed_roc'    : agg['roc_auc'][0],
    'mean_seed_mcc'    : agg['mcc'][0],
}.items()}, indent=2))


## 17b. Zip `runs/` for download (Kaggle -- no Drive)


In [ ]:
_export_names = [
    'checkpoint_vnext.pt', 'scores_vnext.npz', 'metrics_vnext.json',
    'eval_vnext.png', 'drift_report_vnext.npz',
    'metrics_ablation_vnext.json', 'ablation_retrain_table.csv',
    'eval_per_attack_type.png', 'eval_per_attack_roc.png', 'eval_multiclass.json',
    'mdc_label_map.json',
]
present = [n for n in _export_names if (RUN_DIR / n).is_file()]
if not present:
    raise FileNotFoundError(f'{RUN_DIR} has none of {_export_names} -- run \u00a717 save first')
print('Present:', present)

zip_and_offer_download(RUN_DIR, 'model_vnext_runs')


## 18. Comparison vs prior runs
Cross-references this run against historic results so the thesis can quote a single table.

In [ ]:
history_table = [
    ('v0  mdc_model_training', 0.6415, None,   None,   '54+1 feat, RobustScaler, no bucket scaler'),
    ('v2  default',            0.5425, 0.3305, 0.8092, 'mean+max+std, mean-pool decoder'),
    ('v2  HPO retrain',        0.6407, 0.4766, 0.8388, 'mean+max+std, HPO'),
    ('v3  run 2 (Robust)',     None,   None,   None,   'BAD scale (MSE ~1e11)'),
    ('v3  run 3 (Exp1 56f)',   0.1582, -0.1666,0.5345, 'mean-only, inverted'),
    ('v3  run 4 (Exp A)',      0.7163, 0.5278, 0.7105, 'best Exp A default'),
    ('vNext default',          all_results['f1_optimal']['roc_auc'],
                               all_results['f1_optimal']['mcc'],
                               all_results['f1_optimal']['f1'],
                               '+contractive, +ensemble, +F1-thr'),
    ('vNext HPO',              r_best['roc_auc'], r_best['mcc'], r_best['f1'],
                               'HPO best'),
]
print(f'{"run":<25} {"ROC-AUC":>8} {"MCC":>8} {"F1":>8}  notes')
print('-'*78)
for row in history_table:
    name, roc, mcc, f1, notes = row
    rs = f'{roc:>8.4f}' if isinstance(roc,(int,float)) else f'{"-":>8}'
    ms = f'{mcc:>8.4f}' if isinstance(mcc,(int,float)) else f'{"-":>8}'
    fs = f'{f1 :>8.4f}' if isinstance(f1 ,(int,float)) else f'{"-":>8}'
    print(f'{name:<25} {rs} {ms} {fs}  {notes}')

## 18b. Controlled retrain ablations (Task 7 / WBS §3.2)

Research-grade ablations isolate **one** design choice at a time against the default
vNext config (seed=`MAIN_SEED`, same data, same F1-optimal threshold protocol).

| Ablation | Change | Research question |
|----------|--------|-------------------|
| `no_contractive` | `contractive_lambda=0` | Does contractive regularization improve separability? |
| `no_early_stop` | `auc_early_stop=False`, train to `MAX_EPOCHS` | Does AUC early-stop prevent overfit / help restore? |

**Protocol**
1. Same architecture / loader / seed as default training
2. Evaluate with the same `thresholds_from_val` → `f1_optimal` rule as §10
3. Report ΔROC / ΔF1 / ΔMCC vs locked default (**0.7402 / 0.7412 / 0.5884**)
4. Save `metrics_ablation_vnext.json` + `ablation_retrain_table.csv`

Set `RUN_ABLATIONS = True` on Colab GPU (~30–45 min each).  
`SMOKE_ABLATIONS = True` runs 2-epoch harness check only (not thesis numbers).


In [ ]:
# Task 7 — Controlled retrain ablations
RUN_ABLATIONS = True    # ISS-08 fix: was False (placeholder table); True runs real ablations on Colab GPU (~1 hr)
SMOKE_ABLATIONS = False  # True = 2-epoch harness only (NOT for thesis)
ABLATION_SEED = MAIN_SEED

# Locked default reference (mdc_model_vNext_lat_output / metrics_vnext.json)
_metrics_ref = globals().get('metrics', {})
_def = _metrics_ref.get('test_default', {}).get('f1_optimal', {}) if isinstance(_metrics_ref, dict) else {}
DEFAULT_REF = {
    'roc_auc': float(_def.get('roc_auc', 0.7402)),
    'f1': float(_def.get('f1', 0.7412)),
    'mcc': float(_def.get('mcc', 0.5884)),
    'precision': float(_def.get('precision', float('nan'))),
    'recall': float(_def.get('recall', float('nan'))),
    'fpr': float(_def.get('fpr', float('nan'))),
}

def _eval_ablation_model(m, label):
    """Same operating-point protocol as §10 (F1-optimal on val, eval on test)."""
    s_val_a = compute_scores(X_val, feat_std=feat_std, _model=m)
    s_test_a = compute_scores(X_test, feat_std=feat_std, _model=m)
    thr = thresholds_from_val(s_val_a, y_val)['f1_optimal']
    pred = (s_test_a >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        'label': label,
        'val': {
            'roc_auc': float(roc_auc_score(y_val, s_val_a)),
            'thr_f1_optimal': float(thr),
        },
        'test': {
            'roc_auc': float(roc_auc_score(y_test, s_test_a)),
            'pr_auc': float(average_precision_score(y_test, s_test_a)),
            'f1': float(f1_score(y_test, pred, zero_division=0)),
            'mcc': float(matthews_corrcoef(y_test, pred)),
            'precision': float(tp / max(tp + fp, 1)),
            'recall': float(tp / max(tp + fn, 1)),
            'fpr': float(fp / max(fp + tn, 1)),
            'thr': float(thr),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        },
        'delta_vs_default': {
            'roc_auc': float(roc_auc_score(y_test, s_test_a) - DEFAULT_REF['roc_auc']),
            'f1': float(f1_score(y_test, pred, zero_division=0) - DEFAULT_REF['f1']),
            'mcc': float(matthews_corrcoef(y_test, pred) - DEFAULT_REF['mcc']),
        },
    }

def run_ablation(name, **train_kwargs):
    torch.manual_seed(ABLATION_SEED); np.random.seed(ABLATION_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(ABLATION_SEED)
    m = build_model(n_features)
    # Smoke mode: tiny epoch budget to validate harness only
    if SMOKE_ABLATIONS:
        train_kwargs = {**train_kwargs, 'max_epochs': 2, 'min_epoch': 1,
                        'auc_patience': 1, 'verbose': True}
        if 'auc_early_stop' not in train_kwargs:
            train_kwargs['auc_early_stop'] = True
    result, hist = train_one(m, train_loader_base, X_val_t, y_val, verbose=True, **train_kwargs)
    # Restore AUC-best state when available (matches default training restore)
    if result.get('auc_state') is not None:
        m.load_state_dict(result['auc_state'])
    metrics_ab = _eval_ablation_model(m, name)
    metrics_ab['train'] = {
        'best_val_auc': float(result['best_val_auc']),
        'best_val_mse': float(result['best_val_mse']),
        'final_epoch': int(result['final_epoch']),
        'restored': result['restored'],
        'history_len': len(hist),
        'smoke': bool(SMOKE_ABLATIONS),
    }
    metrics_ab['train_kwargs'] = {k: (bool(v) if isinstance(v, bool) else v)
                                  for k, v in train_kwargs.items()}
    metrics_ab['protocol'] = {
        'seed': int(ABLATION_SEED),
        'threshold_rule': 'f1_optimal_on_val',
        'score_invert': bool(SCORE_INVERT),
        'default_ref': DEFAULT_REF,
        'thesis_usable': (not SMOKE_ABLATIONS) and RUN_ABLATIONS,
    }
    del m; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics_ab

ablation_specs = {
    'no_contractive': dict(contractive_lambda=0.0),
    'no_early_stop': dict(auc_early_stop=False, max_epochs=MAX_EPOCHS),
}

ablation_results = {}
_run = RUN_ABLATIONS or SMOKE_ABLATIONS
if _run:
    for name, kw in ablation_specs.items():
        print(f'\n=== Ablation: {name}  smoke={SMOKE_ABLATIONS} ===')
        ablation_results[name] = run_ablation(name, **kw)
        print('test:', json.dumps(ablation_results[name]['test'], indent=2))
        print('delta_vs_default:', json.dumps(ablation_results[name]['delta_vs_default'], indent=2))
    out_path = RUN_DIR / 'metrics_ablation_vnext.json'
    out_path.write_text(json.dumps(ablation_results, indent=2))
    print(f'\nSaved ablations -> {out_path}')

    # Thesis-ready comparison table (offline detector configs only)
    import pandas as pd
    rows = [{
        'config': 'vNext default (reference)',
        'roc_auc': DEFAULT_REF['roc_auc'],
        'f1': DEFAULT_REF['f1'],
        'mcc': DEFAULT_REF['mcc'],
        'delta_roc': 0.0, 'delta_f1': 0.0, 'delta_mcc': 0.0,
        'source': 'metrics_vnext.json',
        'thesis_usable': True,
    }]
    for name, payload in ablation_results.items():
        t = payload['test']; d = payload['delta_vs_default']
        rows.append({
            'config': name,
            'roc_auc': t['roc_auc'], 'f1': t['f1'], 'mcc': t['mcc'],
            'delta_roc': d['roc_auc'], 'delta_f1': d['f1'], 'delta_mcc': d['mcc'],
            'final_epoch': payload['train']['final_epoch'],
            'best_val_auc': payload['train']['best_val_auc'],
            'source': '§18b retrain',
            'thesis_usable': payload['protocol']['thesis_usable'],
        })
    abl_df = pd.DataFrame(rows)
    abl_csv = RUN_DIR / 'ablation_retrain_table.csv'
    abl_df.to_csv(abl_csv, index=False)
    print('\n=== Ablation retrain table (Task 7/8) ===')
    print(abl_df.to_string(index=False))
    print(f'Saved -> {abl_csv}')
    # No Colab/Drive mirror-copy needed on Kaggle -- RUN_DIR is already the
    # single working directory (/kaggle/working/runs).
    if SMOKE_ABLATIONS:
        print('WARNING: SMOKE_ABLATIONS=True — numbers are NOT thesis-usable.')
else:
    print('RUN_ABLATIONS=False — skip retrain. Set True on Colab GPU for thesis ablations.')
    _existing = RUN_DIR / 'metrics_ablation_vnext.json'
    if _existing.is_file():
        ablation_results = json.loads(_existing.read_text())
        print(f'Loaded existing ablations: {list(ablation_results.keys())}')
        for name, payload in ablation_results.items():
            usable = payload.get('protocol', {}).get('thesis_usable', True)
            print(f"  {name}: ROC={payload['test']['roc_auc']:.4f}  "
                  f"F1={payload['test']['f1']:.4f}  usable={usable}")


## 19. Per-attack-type evaluation (Task 10)

Uses in-memory `s_test` / `y_test` from this run.
Loads `y_test_multiclass` from `windows_vnext_mc.npz`.

**Label names** come from the official MDC dataset card (Sever & Dogan, 2023) —
not generic `Attack_N` placeholders. Sparse labels in the reference test split:
0, 1, 2, 3, 4, 8, 11 (others absent).


In [ ]:
MC_NPZ = kaggle_find('windows_vnext_mc.npz')
if MC_NPZ is None:
    print("windows_vnext_mc.npz not found under /kaggle/input or /kaggle/working.")
    print("Attach mdc_preprocess_vNext_mc_kaggle.ipynb's output via '+ Add Data', or upload it now:")
    MC_NPZ = kaggle_upload_fallback('windows_vnext_mc.npz', DATA_DIR)

if MC_NPZ is None:
    raise FileNotFoundError(
        "windows_vnext_mc.npz not found. Run mdc_preprocess_vNext_mc_kaggle.ipynb first "
        "and attach its output via '+ Add Data', or upload it manually.")

print(f'MC_NPZ: {MC_NPZ}')
_mc = np.load(MC_NPZ, allow_pickle=True)
y_test_mc = _mc['y_test_multiclass'].astype(int)
y_test_mc_bin = _mc['y_test'].astype(np.int64)
_mc.close()

assert len(y_test_mc) == len(y_test), f'length mismatch mc={len(y_test_mc)} test={len(y_test)}'
_n_bad = int((y_test_mc_bin != y_test).sum())
if _n_bad:
    raise ValueError(f'{_n_bad} y_test mismatches -- rerun mc preprocess with RANDOM_STATE=42')
print('Alignment: PASS (y_test matches windows_vnext_mc.npz)')
print(f'Unique multiclass labels on test: {sorted(np.unique(y_test_mc).tolist())}')


In [ ]:
from collections import Counter
from sklearn.metrics import roc_auc_score, roc_curve

# Task 10 — official MDC label names (embedded; no separate mdc_label_map.py)
LABEL_NAMES = {
    0: 'BENIGN',
    1: 'CVE-2020-13379',
    2: 'Node-RED Recon',
    3: 'Node-RED RCE',
    4: 'Node-RED Escape',
    5: 'CVE-2021-43798',
    6: 'CVE-2019-20933',
    7: 'CVE-2021-30465',
    8: 'CVE-2021-25741',
    9: 'CVE-2022-23648',
    10: 'CVE-2019-5736',
    11: 'DSB Nuclei Scan',
}
LABEL_NAMES_FULL = {
    0: 'Benign',
    1: 'CVE-2020-13379 (Grafana SSRF / path traversal family)',
    2: 'Node-RED Reconnaissance',
    3: 'Node-RED Remote Code Execution',
    4: 'Node-RED Container Escape',
    5: 'CVE-2021-43798 (Grafana path traversal)',
    6: 'CVE-2019-20933 (InfluxDB auth bypass)',
    7: 'CVE-2021-30465 (runc mount race / escape)',
    8: 'CVE-2021-25741 (Kubernetes symlink exchange)',
    9: 'CVE-2022-23648 (containerd volume mount)',
    10: 'CVE-2019-5736 (runc overwrite / escape)',
    11: 'DSB Nuclei Scan',
}
LABEL_CITATION = (
    'Sever, Y., & Dogan, A. H. (2023). A Kubernetes dataset for misuse detection. '
    'ITU Journal on Future and Evolving Technologies, 4(2), 383-388.'
)

THRESHOLD   = float(thresholds['f1_optimal'])
overall_auc = float(roc_auc_score(y_test, s_test))
pred_test   = (s_test >= THRESHOLD).astype(int)
_mc_out = RUN_DIR
_mc_out.mkdir(parents=True, exist_ok=True)

# Persist label map next to metrics (thesis evidence)
(_mc_out / 'mdc_label_map.json').write_text(json.dumps({
    'label_names': {str(k): v for k, v in LABEL_NAMES.items()},
    'label_names_full': {str(k): v for k, v in LABEL_NAMES_FULL.items()},
    'citation': LABEL_CITATION,
}, indent=2))

rows = []
benign_mask = y_test_mc == 0
n_benign    = int(benign_mask.sum())
fp_benign   = int(pred_test[benign_mask].sum())
fpr_benign  = fp_benign / n_benign if n_benign else 0.0
rows.append({'label': 0, 'name': LABEL_NAMES[0], 'n_windows': n_benign,
             'detected': fp_benign, 'recall_fpr': fpr_benign, 'note': 'FPR'})

present_attack_labels = sorted(l for l in np.unique(y_test_mc) if l != 0)
absent_labels = sorted(set(range(1, 12)) - set(present_attack_labels))

for lbl in present_attack_labels:
    mask = y_test_mc == lbl
    n_win = int(mask.sum())
    tp    = int(pred_test[mask].sum())
    rows.append({'label': int(lbl), 'name': LABEL_NAMES.get(lbl, f'Attack_{lbl}'),
                 'n_windows': n_win, 'detected': tp,
                 'recall_fpr': tp / n_win if n_win else 0.0, 'note': 'Recall'})

print(f'{"Label":>6}  {"Name":<22}  {"Windows":>8}  {"Detected":>9}  {"Recall/FPR":>11}  Note')
print('-' * 75)
for r in rows:
    star = ' *' if r['label'] == 0 else ''
    print(f'{r["label"]:>6}  {r["name"]:<22}  {r["n_windows"]:>8,}  '
          f'{r["detected"]:>9,}  {r["recall_fpr"]:>11.4f}  {r["note"]}{star}')
print(f'\nThreshold: {THRESHOLD:.4f} (f1_optimal)  |  ROC-AUC: {overall_auc:.4f}')
print(f'Present attack labels: {present_attack_labels}')
print(f'Absent in this test split: {absent_labels}')
print(f'Citation: {LABEL_CITATION}')

# Bar chart
atk_rows = [r for r in rows if r['label'] != 0]
fig, ax = plt.subplots(figsize=(max(8, len(atk_rows) * 1.4), 5))
recalls = [r['recall_fpr'] for r in atk_rows]
names   = [r['name'] for r in atk_rows]
n_wins  = [r['n_windows'] for r in atk_rows]
colors  = ['#2ecc71' if r >= 0.80 else '#f39c12' if r >= 0.50 else '#e74c3c' for r in recalls]
bars = ax.bar(range(len(names)), recalls, color=colors, edgecolor='white', linewidth=0.5)
for bar, r, n in zip(bars, recalls, n_wins):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{r:.2f}\n(n={n})', ha='center', va='bottom', fontsize=8)
ax.axhline(fpr_benign, color='red', linestyle='--', linewidth=1.2,
           label=f'FPR on benign = {fpr_benign:.3f}')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Recall'); ax.set_ylim(0, 1.15)
ax.set_title(f'Per-Attack-Type Recall  (thr={THRESHOLD:.4f}, AUC={overall_auc:.4f})')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
_mc_plot = _mc_out / 'eval_per_attack_type.png'
plt.savefig(_mc_plot, dpi=300); plt.savefig(_mc_out / 'eval_per_attack_type_140.png', dpi=140)
plt.show()
print(f'Saved -> {_mc_plot}')

# Per-attack ROC overlay
fig, ax = plt.subplots(figsize=(8, 5))
fpr_b, tpr_b, _ = roc_curve(y_test, s_test)
ax.plot(fpr_b, tpr_b, 'k-', lw=2, label=f'Overall (AUC={overall_auc:.4f})')
per_attack_auc = {}
for lbl in present_attack_labels:
    mask = (y_test_mc == lbl) | (y_test_mc == 0)
    _y = (y_test_mc[mask] != 0).astype(int)
    _s = s_test[mask]
    if _y.sum() == 0 or _y.sum() == len(_y):
        continue
    try:
        _auc = float(roc_auc_score(_y, _s))
        per_attack_auc[int(lbl)] = _auc
        _fp, _tp, _ = roc_curve(_y, _s)
        ax.plot(_fp, _tp, lw=1.2, alpha=0.8,
                label=f'{LABEL_NAMES.get(lbl, lbl)} (AUC={_auc:.3f})')
    except Exception:
        pass
ax.plot([0, 1], [0, 1], 'k:', lw=0.8)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Per-Attack-Type ROC Curves')
ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
_mc_roc = _mc_out / 'eval_per_attack_roc.png'
plt.savefig(_mc_roc, dpi=300); plt.savefig(_mc_out / 'eval_per_attack_roc_140.png', dpi=140)
plt.show()
print(f'Saved -> {_mc_roc}')

# JSON export
mc_results = {
    'threshold': THRESHOLD,
    'overall_roc_auc': overall_auc,
    'fpr_on_benign': fpr_benign,
    'label_names': {str(k): v for k, v in LABEL_NAMES.items()},
    'citation': LABEL_CITATION,
    'present_attack_labels': [int(x) for x in present_attack_labels],
    'absent_attack_labels': [int(x) for x in absent_labels],
    'per_attack_auc_vs_benign': per_attack_auc,
    'per_attack_type': [
        {'label': r['label'], 'name': r['name'], 'n_windows': r['n_windows'],
         'detected': r['detected'],
         'recall': r['recall_fpr'] if r['label'] != 0 else None,
         'fpr': r['recall_fpr'] if r['label'] == 0 else None,
         'auc_vs_benign': per_attack_auc.get(r['label'])}
        for r in rows],
    'interpretation_note': (
        'Sparse labels (n=1) are reported for completeness but are not statistically '
        'reliable; emphasize Attack classes with n>=50 (CVE-2020-13379, Node-RED Recon).'
    ),
}
_mc_json = _mc_out / 'eval_multiclass.json'
_mc_json.write_text(json.dumps(mc_results, indent=2))
print(f'Saved -> {_mc_json}')
_mc_files = [_mc_plot, _mc_roc, _mc_json, _mc_out / 'mdc_label_map.json']
print('Saved (will be included in the §17b runs/ zip):')
for f in _mc_files:
    if f.is_file():
        print(f'  {f.name:30s} {f.stat().st_size:>10,} bytes')
